# 🚀 Distill a tool-using agent with sequence-level knowledge distillation on Amazon SageMaker AI

Sequence-level knowledge distillation of a multi-turn **shopping agent** into `Qwen/Qwen3.5-4B`, trained
with the serverless **`SFTTrainer`** from the SageMaker Python SDK v3. A stronger teacher's tool-use trajectories become
supervised fine-tuning data; the student learns to produce the same next action given the same context.

### The task: ShoppingBench
[ShoppingBench](https://github.com/yjwjy/ShoppingBench) ([paper](https://arxiv.org/abs/2508.04266)) is an end-to-end
benchmark for e-commerce agents. A shopper makes a natural-language request, for example *"women's jeans in size eu:30
with a high waist, priced above 114 PHP"*, and the agent must find a product (or set of products) in a catalog of ~2.7M real
products that satisfies every stated constraint.

- **Intents:** single-product search, finding one shop that sells several products, shopping within a voucher and a
  budget, and a knowledge intent that needs web search.
- **Tools:** `find_product` (catalog search with price, service and sort filters), `view_product_information`
  (attributes and SKU options), `recommend_product` (the answer) and `terminate`.
- **The agent loop:** at every step the model receives the dialogue so far, emits its next tool call as text, the
  harness executes it against a local search server and appends the result, until the agent terminates.
- **Scoring:** rule-based and reproducible. A problem counts as solved (**ASR**, its success rate) only if the
  recommended product meets every constraint of the request: product match, price range, services, SKU options and
  attributes, plus the same-shop or budget rule for the multi-product intents.

### The data
The teacher trajectories come from `oro-ai/sn15-shoppingbench-sft-15k`: ShoppingBench runs by agents competing on ORO's
Bittensor subnet, filtered to those where the model itself issued the tool calls and judged by an LLM for reasoning
quality.

### What this notebook does

| Step | |
|---|---|
| 01 | Encode each trajectory step as a `{prompt, completion}` pair in the **exact** format the benchmark's agent loop sends and parses |
| 02 | Sequence-level knowledge distillation = SFT with the serverless `SFTTrainer` |
| 03 | Collect the registered Model Package |
| 04 | LLM-as-a-judge (managed): is each next action a good one, given the context? |

**Task success (ASR) and instruction-following regression (IFEval)** are measured in the companion notebook
`evaluate-tool-call-accuracy.ipynb`, which deploys the base and distilled models to SageMaker real-time endpoints and
runs them through the benchmark. Reference results from a run of this recipe (LoRA r64, held-out test problems, thinking on), base → student ASR:
**product 0.379 → 0.466, shop 0.114 → 0.343, voucher 0.095 → 0.129** (mean of the three: 0.196 → 0.313).

## 00. Setup

The serverless path lives in SDK **v3** namespaces: `sagemaker.train.sft_trainer.SFTTrainer`,
`sagemaker.ai_registry.dataset.DataSet`, and `sagemaker.core.resources`.

In [ ]:
import os, json, boto3
from sagemaker.core.helper.session_helper import get_execution_role

REGION = boto3.Session().region_name or "us-west-2"
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"] = REGION

# Needed for native MLflow metrics while the trainer waits:
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = f"https://mlflow.sagemaker.{REGION}.app.aws"

try:
    ROLE = get_execution_role()
    print("Execution role:", ROLE)
except Exception:
    ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
    ROLE = f"arn:aws:iam::{ACCOUNT}:role/<your-sagemaker-execution-role>"   # <- replace

BUCKET  = "<your-bucket>"                             # <- replace with your bucket
PREFIX  = "seqkd-shopping"
S3_ROOT = f"s3://{BUCKET}/{PREFIX}"
print(REGION, ROLE, S3_ROOT)

### Thinking mode (shared by training and evaluation)

The ORO trajectories carry no reasoning text (`<think>` was stripped upstream), so the student learns **actions only**.
The trainer renders each training example with Qwen3.5's chat template, which writes an assistant turn without
reasoning as `<think>\n\n</think>\n\n<tool_call>[…]</tool_call><|im_end|>`: the **thinking-off** form. Evaluate the
student the same way: set `ENABLE_THINKING = False` in `evaluate-tool-call-accuracy.ipynb`, which sends
`extra_body={"chat_template_kwargs": {"enable_thinking": False}}` with every request.

To distill reasoning as well, the teacher trajectories must contain it (for example, regenerate them with a model that
writes `<think>` text) and the assistant messages must keep it.

In [ ]:
ENABLE_THINKING = False   # must match ENABLE_THINKING in evaluate-tool-call-accuracy.ipynb

## 01. Data prep

### 01a. Load the teacher trajectories and identify the held-out test problems

Public ShoppingBench trajectory sets can be built on the benchmark's own test queries: this corpus covers most of the
product, shop and voucher test queries. A score on those problems would partly measure recall, so the evaluation
notebook scores only **held-out** test problems, whose query never appears in this corpus. Training on the other
queries cannot leak into that score, so all trajectories are kept (dropping every test query would leave ~100 of the
15k trajectories).

In [ ]:
import json, random
from huggingface_hub import list_repo_files, hf_hub_download
from transformers import AutoTokenizer
HF_STUDENT="Qwen/Qwen3.5-4B"
REPO = "oro-ai/sn15-shoppingbench-sft-15k"   # example teacher trajectories
jsonls = [f for f in list_repo_files(REPO, repo_type="dataset") if f.endswith(".jsonl")]
sft_file = next(f for f in jsonls if "metadata" not in f.lower() and "eval" not in f.lower())
raw = [json.loads(l) for l in open(hf_hub_download(REPO, sft_file, repo_type="dataset")) if l.strip()]
raw = [r for r in raw if r.get("messages")]
print(len(raw), "trajectories")


# ShoppingBench code and test files at the pinned commit, in the same folder evaluate-tool-call-accuracy.ipynb uses (its
# WORK), so both notebooks share one checkout. --code-only skips the catalog, Java and index (only evaluation needs them).
import subprocess
from pathlib import Path
SHOPBENCH_WORK = Path.home() / "shopbench-eval"
subprocess.run(["bash", "scripts/setup_shopbench.sh", "--code-only", str(SHOPBENCH_WORK)], check=True)
SB_DIR = SHOPBENCH_WORK / "ShoppingBench"
test_rows = {t: [json.loads(l) for l in open(SB_DIR / "data" / f"synthesize_{t}_test.jsonl") if l.strip()]
             for t in ("product", "shop", "voucher")}

def traj_query(t):                      # a trajectory's task == its first user message
    return next((m["content"] for m in t["messages"] if m.get("role") == "user"), None)

train_queries = {traj_query(t) for t in raw}
for t, rows in test_rows.items():
    n = sum(r["query"] not in train_queries for r in rows)
    print(f"{t}: {n} / {len(rows)} test problems are held out (query not in the corpus)")

### 01b. Encode every step in the ShoppingBench harness format

One training example per tool-calling step. The student is evaluated by ShoppingBench's agent loop, which at every
step sends two messages and parses the reply, so each example reproduces exactly that, built with the harness's own
functions:

| Message | Content |
|---|---|
| `user` | `rollout.md` (the agent instructions and tool list), then the dialogue so far: the shopper's request plus every earlier step as `<user>` / `<tool_call>` / `<obs>` blocks |
| `assistant` (the target) | The teacher's next action: `<tool_call>[{"name": …, "parameters": …}]</tool_call>` |

The harness sends `rollout.md` as a separate system message. Here it opens the user message instead, because the
serverless trainer renders each message on its own to build the loss mask, and Qwen3.5's chat template rejects a system
message rendered without a user message after it (the job fails with "System message must be at the beginning"). The
text the model sees is the same; only the role header around the instructions differs.

Examples use the documented `messages` format with **plain strings**. The trainer applies Qwen3.5's chat template and
computes the loss on the assistant message only, so the prompt is not pre-rendered here (a pre-rendered prompt would be
wrapped in the template a second time). Tool arguments are normalized to what the harness tools accept (for example
`product_ids` as one comma-separated string), steps using tools the harness does not have are dropped, and the
train/validation split is by query. A final self-check runs every target through the harness's parser.

In [ ]:
# --- ENCODE IN THE SHOPPINGBENCH HARNESS FORMAT -------------------------------------------------
# The student is evaluated by the benchmark's agent loop, which at EVERY step sends exactly two messages
# (system = rollout.md with the tool list; user = "# Dialogue Records History" + every earlier step as
# <user>/<tool_call>/<obs> blocks) and parses the reply with util/message.py, which only accepts
# <tool_call>[{"name": ..., "parameters": {...}}]</tool_call>. Train on exactly that, using the harness's
# own functions so the strings are byte-identical. (Native multi-turn chat with tools= would teach Qwen's
# XML tool-call format, which the parser silently drops.)
import subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ujson", "pydantic", "portalocker", "openai",
                "duckduckgo_search", "googlesearch-python", "mcp", "colorama"], check=True)
sys.path.insert(0, str(SB_DIR / "src" / "agent"))
from run_rollout import get_system_prompt, get_user_prompt
from util.message import Message, generate_tool_call_id

SYSTEM_PROMPT = get_system_prompt({"system_prompt_file": str(SB_DIR / "src/agent/prompt/rollout.md"),
                                   "exclude_tools": ["web_search"]})   # web search is only used by the knowledge intent
HARNESS_TOOLS = {"find_product", "view_product_information", "recommend_product", "terminate"}
tok = AutoTokenizer.from_pretrained(HF_STUDENT, trust_remote_code=True)

def harness_call(tc):
    """ORO tool call -> the argument shapes the harness tools accept."""
    f = tc.get("function", tc); name = f["name"]; a = f.get("arguments") or {}
    a = json.loads(a) if isinstance(a, str) else dict(a)
    if name == "find_product":
        if "q" not in a and "query" in a: a["q"] = a.pop("query")
        a.setdefault("page", 1)                                   # required by the harness's find_product
    elif name in ("view_product_information", "recommend_product"):
        v = a.pop("product_ids", a.pop("product_id", ""))
        a["product_ids"] = ",".join(map(str, v)) if isinstance(v, list) else str(v)   # comma-separated string
    elif name == "terminate":
        s = a.get("status", "success"); a["status"] = s if isinstance(s, str) else ("success" if s else "failure")
    return {"name": name, "parameters": a}

def to_rows(traj):
    msgs = [m for m in traj["messages"] if m.get("role") != "system"]
    query = next(m["content"] for m in msgs if m["role"] == "user")
    steps, i = [], 0
    while i < len(msgs):                     # each tool-calling assistant turn + the tool results after it
        m = msgs[i]; i += 1
        if m["role"] != "assistant" or not m.get("tool_calls"):
            continue                         # prose-only turns ("Continuing analysis...", final "Done.") never occur in the harness loop
        calls = [harness_call(tc) for tc in m["tool_calls"]]
        if any(c["name"] not in HARNESS_TOOLS for c in calls):
            return None                      # uses a tool the ShoppingBench harness does not have
        results = []
        while i < len(msgs) and msgs[i]["role"] == "tool":
            r = msgs[i]["content"]
            try: r = json.loads(r)
            except Exception: pass
            results.append(r); i += 1
        steps.append((calls, results))
    if not steps:
        return None
    rows, history, prev = [], [], Message(user=query)
    for calls, results in steps:
        user_prompt = get_user_prompt(prev, history)               # appends prev to history, as the agent loop does
        # The system prompt goes at the top of the user message: the serverless trainer renders each message on its
        # own, and Qwen3.5's chat template rejects a system message rendered without a following user message.
        rows.append({"messages": [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_prompt}"},
                                  {"role": "assistant", "content": f"<tool_call>{json.dumps(calls, ensure_ascii=False)}</tool_call>"}]})
        tcs = [dict(c, tool_call_id=generate_tool_call_id(c["name"], c["parameters"])) for c in calls]
        prev = Message(tool_call=tcs, obs=[{"tool_call_id": t["tool_call_id"], "results": r} for t, r in zip(tcs, results)])
        if any(c["name"] == "terminate" for c in calls):
            break                                                   # the harness loop stops here
    return rows

# Split by QUERY, not by step: ~30 trajectories share each query, so a step-level split leaks.
by_q = {}
for t in raw: by_q.setdefault(traj_query(t), []).append(t)
queries = sorted(by_q); random.seed(42); random.shuffle(queries)
val_q = set(queries[:max(1, len(queries) // 20)])                  # ~5% of queries held out
train, val, skipped = [], [], 0
for q, trajs in by_q.items():
    for t in trajs:
        rows = to_rows(t)
        if rows is None: skipped += 1; continue
        (val if q in val_q else train).extend(rows)
random.shuffle(train)
print(f"{len(raw)} trajectories ({skipped} skipped: non-harness tools or no tool calls) -> "
      f"train {len(train)} / val {len(val)} examples; {len(queries) - len(val_q)} / {len(val_q)} queries")

# Self-check with the harness's own parser: every target must yield a tool call.
target = lambda row: row["messages"][-1]["content"]
rate = sum(bool(Message.from_string("", target(r)).tool_call) for r in train[:2000]) / min(len(train), 2000)
print(f"harness parser accepts {rate:.1%} of targets"); assert rate > 0.99

# Length filter: the trainer cuts sequences longer than dataset_max_len (step 02) from the LEFT, which would remove the
# system prompt. Drop those few examples instead.
MAX_TOKENS = 16384                                                  # = dataset_max_len in step 02
n_tokens = lambda r: len(tok(tok.apply_chat_template(r["messages"], tokenize=False))["input_ids"])
for name, rows in (("train", train), ("val", val)):
    lens = [n_tokens(r) for r in rows]
    keep = [r for r, n in zip(rows, lens) if n <= MAX_TOKENS]
    s = sorted(lens)
    print(f"{name}: tokens per example median {s[len(s) // 2]}, p99 {s[int(len(s) * 0.99)]}, max {s[-1]}; "
          f"dropped {len(rows) - len(keep)} longer than {MAX_TOKENS}")
    rows[:] = keep
print("--- example (as the trainer renders it) ---\n" + tok.apply_chat_template(train[0]["messages"], tokenize=False))

TRAIN_FILE, VAL_FILE = "seqkd_train.jsonl", "seqkd_val.jsonl"
def write_jsonl(rows, p):
    with open(p, "w") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
write_jsonl(train, TRAIN_FILE); write_jsonl(val, VAL_FILE)

#### Upload to S3, then register the dataset in the SageMaker AI Registry

The serverless `SFTTrainer` consumes a **registered dataset ARN**, not a raw S3 path. `DataSet.create` wraps a JSONL
file in a versioned registry entry. Each line holds one `messages` list, exactly what the cell above wrote.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

s3 = boto3.client("s3", region_name=REGION)
s3.upload_file(TRAIN_FILE, BUCKET, f"{PREFIX}/data/{TRAIN_FILE}")
s3.upload_file(VAL_FILE,   BUCKET, f"{PREFIX}/data/{VAL_FILE}")
TRAIN_S3 = f"{S3_ROOT}/data/{TRAIN_FILE}"
VAL_S3   = f"{S3_ROOT}/data/{VAL_FILE}"

train_dataset = DataSet.create(name="seqkd-shopping-train", source=TRAIN_S3)
val_dataset   = DataSet.create(name="seqkd-shopping-val",   source=VAL_S3)
print("train:", train_dataset.arn)
print("val:  ", val_dataset.arn)

## 02. Fine-tune (Sequence-level knowledge distillation) with serverless `SFTTrainer`

Distillation here *is* ordinary SFT on the teacher trajectories. You give the trainer a base model id, a
training type, the two registered datasets, and an output path.

`model_package_group` is **required** when you pass a base model id — it's where the output Model Package
lands. `accept_eula=True` is required for gated base models.

In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType
from sagemaker.core.resources import ModelPackageGroup

MPG_NAME = "seqkd-qwen35-4b-shopping"
try:
    model_package_group = ModelPackageGroup.create(model_package_group_name=MPG_NAME)
except Exception as e:
    print("reusing existing group:", type(e).__name__)
    model_package_group = MPG_NAME          # an existing name or ARN is also accepted

# To find out which models are supported by serverless training:
# SFTTrainer.list_supported_models()

sft_trainer = SFTTrainer(
    model="huggingface-vlm-qwen3-5-4b",     # Qwen3.5-4B student model
    training_type=TrainingType.LORA,        # TrainingType.FULL for FFT
    model_package_group=model_package_group,
    mlflow_experiment_name="seqkd-shopping-exp",
    training_dataset=train_dataset.arn,
    validation_dataset=val_dataset.arn,
    s3_output_path=f"{S3_ROOT}/output/",
    accept_eula=True,
)

### Inspect, then override the hyperparameters if needed

Print the recipe defaults first and change only what you need.

In [ ]:
from pprint import pprint
print("Recipe defaults:")
pprint(sft_trainer.hyperparameters.to_dict())

In [ ]:
sft_trainer.hyperparameters.dataset_max_len       = 16384
sft_trainer.hyperparameters.max_epochs            = 2
sft_trainer.hyperparameters.global_batch_size     = 64
sft_trainer.hyperparameters.learning_rate         = 0.0001
sft_trainer.hyperparameters.lora_rank             = 64
sft_trainer.hyperparameters.lora_alpha            = 256     # alpha/rank = 4
sft_trainer.hyperparameters.train_val_split_ratio = 0.9
sft_trainer.hyperparameters.lr_warmup_steps_ratio = 0.03
pprint(sft_trainer.hyperparameters.to_dict())

### Dry-run, then train

`dry_run=True` validates IAM permissions, dataset ARNs, hyperparameters, and infrastructure without
submitting a job — cheap insurance before a multi-hour run.

In [ ]:
sft_trainer.train(dry_run=True)
print("Dry run passed - configuration is valid.")

In [ ]:
# training_job = sft_trainer.train(wait=True)

# To submit now and monitor later instead:
training_job = sft_trainer.train(wait=False)
sft_trainer.stream_logs()

print(training_job.training_job_name, training_job.training_job_status)

## 03. Collect the output Model Package

**There is no merge step here, and that is the point.** A self-managed LoRA job writes an *adapter* that
you then have to download, merge into the base weights, re-add the VLM processor configs
(`preprocessor_config.json`, `processor_config.json`), and re-upload. The serverless job registers a
ready-to-deploy **Model Package** instead; the completed job's `output_model_package_arn` points at it.

In [ ]:
from sagemaker.core.resources import TrainingJob, ModelPackage

job = TrainingJob.get(training_job_name=training_job.training_job_name)
MODEL_PKG_ARN = job.output_model_package_arn
print("Output Model Package ARN:", MODEL_PKG_ARN, " <- set this in evaluate-tool-call-accuracy.ipynb")

model_package = ModelPackage.get(model_package_name=MODEL_PKG_ARN)
print("status:", model_package.model_package_status)

# To find the package later without the job object:
#   aws sagemaker list-model-packages --model-package-group-name seqkd-qwen35-4b-shopping \
#     --sort-by CreationTime --sort-order Descending \
#     --query 'ModelPackageSummaryList[0].ModelPackageArn'

## 04. Evaluate

| Question | How | Where |
|---|---|---|
| Does the student solves shopping tasks with higher accuracy | **ASR** on held-out ShoppingBench problems | `evaluate-tool-call-accuracy.ipynb` |
| Does the trained model lose general instruction following | **IFEval** | `evaluate-tool-call-accuracy.ipynb` |
| Is each next action a good one | **LLM-as-a-judge**: managed `LLMAsJudgeEvaluator` on held-out steps (reference-free) | below; fully managed, runs any time after step 03 |

### LLM-as-a-judge: quality of the next action, on held-out steps (managed)

A managed `LLMAsJudgeEvaluator` job sees one prompt and returns one reply, so it cannot run the agent loop. Instead it
gets single **steps** of held-out teacher trajectories (`oro_sft_eval_v0.jsonl`, 75 problems no model here trained on),
sampled from the first, middle and final steps. Each prompt is the agent instructions plus the dialogue so far, with
the real tool results the teacher saw.

SageMaker generates the base and student replies, and Claude rates each next action 1 / 0.5 / 0
(`NextActionQuality`, plus the built-in `Correctness`). The rating is **reference-free**: the managed LLM-as-a-judge
path does not pass a reference answer to the judge, so it grades the action against the request and the tool results
in the prompt, not against the teacher's action. This isolates "given the right context, is the next action good?". It
does not capture errors that pile up over the agent's own steps; ASR does.

The managed job applies the model's default chat template (Qwen3.5: thinking on), so a reply may start with reasoning
ending in `</think>`; the metric tells the judge to ignore it.

**Quick check before the scores arrive.** Each model's replies are written first to
`<s3_output_path>/*Inference-*/.../inference_output.jsonl`. A well-trained student replies with a single tool call
(one or more actions) and stops there, without making up the tool's results.

In [ ]:
# Managed LLM-as-a-judge, base vs student on held-out steps. ~30-60 min, mostly managed inference.
import json, random, boto3
from huggingface_hub import hf_hub_download
from sagemaker.train.evaluate import LLMAsJudgeEvaluator

N_STEPS = 200                    # sampled held-out steps (stratified over first / middle / final steps)
JUDGE = "anthropic.claude-sonnet-4-5-20250929-v1:0"

held_out = [json.loads(l) for l in open(hf_hub_download(REPO, "oro_sft_eval_v0.jsonl", repo_type="dataset")) if l.strip()]

def step_records(traj):
    """The steps of one held-out trajectory, built exactly like the training data (to_rows, step 01b)."""
    rows = to_rows(traj) or []
    out = []
    for k, r in enumerate(rows):
        user, target = (m["content"] for m in r["messages"])
        out.append({"prompt": user, "completion": target,
                    "_pos": "first" if k == 0 else ("final" if k == len(rows) - 1 else "middle")})
    return out

pool = {"first": [], "middle": [], "final": []}
for t in held_out:
    for r in step_records(t): pool[r["_pos"]].append(r)
rng = random.Random(0)
records = []
for pos, share in (("first", 0.25), ("middle", 0.5), ("final", 0.25)):
    records += rng.sample(pool[pos], min(len(pool[pos]), int(N_STEPS * share)))
records = [r for r in records if len(tok(r["prompt"])["input_ids"]) < 12000]      # keep inference well inside context
print({p: len(v) for p, v in pool.items()}, "->", len(records), "sampled steps")

PERSTEP_FILE = "llmaj_next_step.jsonl"
with open(PERSTEP_FILE, "w") as f:
    for r in records: f.write(json.dumps({"prompt": r["prompt"], "completion": r["completion"]}, ensure_ascii=False) + "\n")
boto3.client("s3", region_name=REGION).upload_file(PERSTEP_FILE, BUCKET, f"{PREFIX}/eval/llmaj/{PERSTEP_FILE}")

# The managed LLM-as-a-judge path does not pass a reference answer to the judge (its records carry
# referenceResponse="dummy"; ground truth is only used by Custom Scorer evaluations), so the metric is reference-free:
# the judge rates the next action against the request and the tool results already in the prompt.
NEXT_ACTION = {"customMetricDefinition": {
    "name": "NextActionQuality",
    "instructions": (
        "You are grading the NEXT ACTION of a shopping agent that uses tools: find_product (search), "
        "view_product_information (inspect candidates), recommend_product (final answer), terminate.\n"
        "The prompt holds the agent instructions, the user's request and the dialogue so far, including real tool results. "
        "Ignore any reasoning text that ends with </think>, any wrapping such as quotes or brackets around the reply, and "
        "differences in formatting; judge only the tool call(s) the response makes.\n"
        "Score 1 if the action is a strong next step: it uses the right tool and its arguments are well grounded in the "
        "user's constraints and the tool results so far (a targeted search, inspecting a promising candidate, or "
        "recommending a product the results support and then terminating). Score 0.5 if it is reasonable but clearly "
        "weaker (a vague search, skipping a needed check, a weaker candidate). Score 0 if it is wrong, premature, repeats "
        "a failed action, invents tool results, or contains no usable tool call.\n\n"
        "Prompt: {{prompt}}\nResponse: {{prediction}}"),
    "ratingScale": [{"definition": "Strong next step", "value": {"floatValue": 1}},
                    {"definition": "Reasonable but weaker", "value": {"floatValue": 0.5}},
                    {"definition": "Wrong or missing", "value": {"floatValue": 0}}]}}
judge = LLMAsJudgeEvaluator(
    model=MODEL_PKG_ARN,                       # the student; evaluate_base_model adds the base model
    evaluator_model=JUDGE,
    dataset=f"{S3_ROOT}/eval/llmaj/{PERSTEP_FILE}",
    builtin_metrics=["Correctness"],           # generic correctness, judged from the prompt (no reference)
    custom_metrics=json.dumps([NEXT_ACTION]),
    s3_output_path=f"{S3_ROOT}/eval/llmaj/out/",
    evaluate_base_model=True,
)
judge.evaluate(dry_run=True); print("dry run passed")
execution = judge.evaluate()
print("execution:", execution.arn)
execution.wait(poll=60, timeout=4 * 3600)
print("status:", execution.status.overall_status)
execution.show_results(limit=20, offset=0, show_explanations=False)